# Equità e bias algoritmico

Il codice del capitolo [«Equità e bias algoritmico»](https://book.paithon.it/main/AIResponsabile/equita-e-bias.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy

## Equità e bias algoritmico

[Leggi la pagina](https://book.paithon.it/main/AIResponsabile/equita-e-bias.html)


### Il conflitto, coi numeri


In [ ]:
import numpy as nprng = np.random.default_rng(0)def genera_gruppo(n, alpha, beta):    # Il punteggio è calibrato per costruzione: P(Y=1 | S=s) = s    s = rng.beta(alpha, beta, size=n)          # punteggio in [0,1]    y = (rng.random(n) < s).astype(int)        # etichetta vera ~ Bernoulli(s)    return s, y# Gruppo A: rischio di base più alto; Gruppo B: più bassosA, yA = genera_gruppo(20000, 3.0, 3.0)        # media score ~0,50sB, yB = genera_gruppo(20000, 2.0, 4.0)        # media score ~0,33soglia = 0.5def tassi(s, y, t):    yhat = (s >= t).astype(int)    sel = yhat.mean()                # selection rate: quota di sì    tpr = yhat[y == 1].mean()        # veri positivi / positivi reali    fpr = yhat[y == 0].mean()        # falsi positivi / negativi reali    ppv = y[yhat == 1].mean()        # valore predittivo positivo (precision)    return sel, tpr, fpr, ppvfor nome, s, y in [("A", sA, yA), ("B", sB, yB)]:    sel, tpr, fpr, ppv = tassi(s, y, soglia)    print(f"Gruppo {nome}: base={y.mean():.3f}  selection={sel:.3f}  "          f"TPR={tpr:.3f}  FPR={fpr:.3f}  VPP={ppv:.3f}")# Calibrazione per gruppo: in ogni bin di score, frazione reale di positivibins = np.linspace(0, 1, 6)print("\nCalibrazione (bin di score -> frazione reale di positivi):")for nome, s, y in [("A", sA, yA), ("B", sB, yB)]:    idx = np.clip(np.digitize(s, bins) - 1, 0, len(bins) - 2)    riga = [f"[{bins[b]:.1f},{bins[b+1]:.1f})->{y[idx == b].mean():.2f}"            for b in range(len(bins) - 1)]    print(f"  Gruppo {nome}:", "  ".join(riga))

## Privacy e robustezza: dati protetti e attacchi avversari

[Leggi la pagina](https://book.paithon.it/main/AIResponsabile/privacy-e-robustezza.html)


### Privacy differenziale: rumore calibrato al singolo


In [ ]:
import numpy as nprng = np.random.default_rng(0)def conteggio_privato(conteggio_vero, epsilon):    sensibilita = 1.0                       # un individuo cambia il conteggio di 1    b = sensibilita / epsilon               # scala del rumore di Laplace    return conteggio_vero + rng.laplace(0.0, b)vero = 42stime = [conteggio_privato(vero, epsilon=0.5) for _ in range(5)]print("vero:", vero, " privati:", np.round(stime, 1))# vero: 42  privati: [42.6 40.8 37.  35.2 44. ]

### FGSM in pratica, con NumPy


In [ ]:
import numpy as nprng = np.random.default_rng(0)# --- dataset giocattolo in dimensione d, da un vero modello logistico ---d, n = 30, 500w_true = rng.normal(size=d)X = rng.normal(size=(n, d))prob = 1.0 / (1.0 + np.exp(-(X @ w_true)))y = (rng.random(n) < prob).astype(float)def sigmoid(z):    return 1.0 / (1.0 + np.exp(-z))# --- regressione logistica addestrata con la discesa del gradiente ---w, b = np.zeros(d), 0.0for _ in range(3000):    p = sigmoid(X @ w + b)    w -= 0.2 * (X.T @ (p - y) / n)    b -= 0.2 * np.mean(p - y)# --- un esempio classificato correttamente e con buona confidenza ---i = 1x, yt = X[i].copy(), y[i]p0 = sigmoid(x @ w + b)# --- FGSM: un passo lungo il segno del gradiente della loss rispetto a x ---grad_x = (p0 - yt) * w                      # dL/dx per la cross-entropy logisticaeps = 0.15x_adv = x + eps * np.sign(grad_x)p1 = sigmoid(x_adv @ w + b)print(f"vera etichetta y = {int(yt)}")print(f"originale:  p(classe 1) = {p0:.3f}  ->  predice {int(p0 > 0.5)}  (corretto)")print(f"avversario: p(classe 1) = {p1:.3f}  ->  predice {int(p1 > 0.5)}  (sbagliato)")print(f"perturbazione: {eps} per feature; norma L2 = {np.linalg.norm(x_adv - x):.2f}"      f" contro {np.linalg.norm(x):.2f} dell'input")

## Allineamento e governance: dai valori umani alle regole

[Leggi la pagina](https://book.paithon.it/main/AIResponsabile/allineamento-e-governance.html)


### Il problema dell'allineamento


In [ ]:
import numpy as nprng = np.random.default_rng(0)n = 2000  # risposte candidate a uno stesso prompt# Cio' che ci interessa davvero: quanto la risposta e' utile e corretta (0-1).qualita_vera = rng.uniform(0, 1, n)# Una caratteristica superficiale che i giudizi umani tendono a premiare.lunghezza = rng.uniform(0, 1, n)# Il reward model imita quei giudizi: approssima la qualita' vera, ma con un# bias sistematico verso le risposte lunghe (e un po' di rumore).proxy = qualita_vera + 1.5 * lunghezza + rng.normal(0, 0.1, n)# Ottimizzare il proxy = tenere le risposte col punteggio surrogato piu' alto.top = 20scelte_proxy = np.argsort(proxy)[-top:]          # top-20 secondo il giudicescelte_vere  = np.argsort(qualita_vera)[-top:]   # top-20 secondo l'obiettivo veroprint(f"Qualita' vera media ottimizzando il proxy: {qualita_vera[scelte_proxy].mean():.3f}")print(f"Qualita' vera media ottimizzando il vero:  {qualita_vera[scelte_vere].mean():.3f}")print(f"Lunghezza media delle scelte col proxy:    {lunghezza[scelte_proxy].mean():.3f}")print(f"Lunghezza media su tutte le risposte:      {lunghezza.mean():.3f}")# Qualita' vera media ottimizzando il proxy: 0.907# Qualita' vera media ottimizzando il vero:  0.994# Lunghezza media delle scelte col proxy:    0.931# Lunghezza media su tutte le risposte:      0.494